# 03. `Time_difference` 정의 탐색 · 금액 부호 분리 · 층화 그룹 분할 확정

세 가지 결정을 내리는 구간이다.

1. **`Time_difference_seconds` 음수 156건** → 절댓값이 아니라 **결측(NaN)** 으로. LightGBM 이 결측을 분기로 흡수하게 둔다 (최소 −33년은 비물리적이라 절댓값이 의미를 만들어낸다).
2. **금액 부호 분리** → `Transaction_is_withdrawal` + `Transaction_Amount_abs`. 이 파생이 뒤에 SHAP 1위(0.4471)가 된다.
3. **`StratifiedGroupKFold` 80/20** → 고객 누수 0건 · 계좌 누수 0건 검증.

> **출처** — `원본/FDS_전처리_정리본.ipynb` (109셀 · Colab 실행본)
>
> 원본은 한 노트북에 §0~§15 를 전부 담고 있어 어디까지가 한 덩어리인지 알기 어려웠다.
> 이 저장소는 **절 경계 그대로** 5개로 나누고 **원본 실행 출력 98건을 모두 보존**했다.
> 코드는 손대지 않았다 — 셀 순서·내용 모두 원본과 동일하다.



### 재현 조건

| 항목 | 값 |
|---|---|
| 입력 | `train_final.csv` (원본 120,000행 × 64컬럼) |
| 원본 실행 경로 | `/content/train_final.csv` (Google Colab) |
| 저장소 경로 | `data/train.csv` — 용량(54MB) 때문에 미포함, `data/README.md` 참고 |
| 최종 산출물 | `X_tr/X_va/y_tr/y_va.parquet` (3차 전처리 · 58피처) + `label_encoders.pkl` · `le_target.pkl` |

> 노트북을 **순서대로(01→05)** 실행해야 한다. 앞 노트북의 `train` · `df` · `tr_idx`/`va_idx` 를
> 뒤 노트북이 이어받는 구조라, 단독 실행하면 `NameError` 가 난다.
> 각 노트북 첫 셀에 이어받는 변수를 명시해 두었다.

**변수 인계** — 02 에서 `train`, `train_raw` 를 이어받는다. 만들어 내보내는 것: `df_tr`/`df_va` 와 `tr_idx`/`va_idx`.

> 원본 셀 범위: `[24] ~ [40]` (총 17셀)


## 7. `Time_difference_seconds` 정의 탐색 → 결론 → 처리

In [ ]:
# 7-1. 음수 케이스 점검 (처리 전)
neg_mask = train['Time_difference_seconds'] < 0
print('음수 건수:', int(neg_mask.sum()))
print('음수 최소(초):', train.loc[neg_mask, 'Time_difference_seconds'].min())
print('음수 최소(일 환산):', round(train.loc[neg_mask, 'Time_difference_seconds'].min() / 86400, 1))
display(train.loc[neg_mask, 'Fraud_Type'].value_counts())
display(train.loc[neg_mask, ['Time_difference', 'Time_difference_seconds']].head())

음수 건수: 156
음수 최소(초): -1045256012.0
음수 최소(일 환산): -12097.9


,count
Fraud_Type,
m,154
a,2


,Time_difference,Time_difference_seconds
74,-11381 days +21:39:31,-983240429.0
84,-7993 days +05:48:41,-690574279.0
111,-8775 days +10:23:42,-758122578.0
208,-1971 days +07:08:21,-170268699.0
224,-1536 days +15:53:02,-132653218.0


In [ ]:
# 7-4. 분포 + 유형별 중앙값 (피처 가치 근거)
display(train['Time_difference_seconds'].describe(percentiles=[.01, .05, .5, .95, .99]))
display(train.groupby('Fraud_Type')['Time_difference_seconds'].median())

,Time_difference_seconds
count,1.200000e+05
mean,7.578565e+05
std,2.003843e+07
min,-1.045256e+09
1%,2.250000e+02
5%,9.789500e+02
50%,9.169500e+03
95%,5.884124e+05
99%,3.198631e+07
max,3.214080e+07


,Time_difference_seconds
Fraud_Type,
a,3222.0
b,8815.0
c,8451.5
d,9166.0
e,8594.5
f,7411.5
g,8669.0
h,161564.0
i,9403.0


### 결론

음수 156건(대부분 정상 m, 최소 ≈-33년)은 비물리적이므로 **절댓값 대신 결측(NaN)** 처리 → LightGBM이 결측을 분기로 흡수.
**주의: `test.csv` 처리 시에도 동일하게 '음수 → NaN' 을 적용해야 일관성이 유지된다.**

In [ ]:
# 7-5. 결정 적용: 음수 → NaN, 중복인 timedelta 원본 컬럼 드롭
train.loc[train['Time_difference_seconds'] < 0, 'Time_difference_seconds'] = np.nan
print('음수 → NaN 처리 후 결측 수:', int(train['Time_difference_seconds'].isna().sum()))

if 'Time_difference' in train.columns:
    train = train.drop(columns=['Time_difference'])
print('남은 컬럼 수:', train.shape[1])

음수 → NaN 처리 후 결측 수: 156
남은 컬럼 수: 64


### 7-6. datetime 컬럼 재검증 (오분류 재발 방지 재확인)

In [ ]:
# 8-1. datetime 으로 잡힌 컬럼이 실제로 날짜형인지 (오분류 재발 방지)
print('=== datetime_cols 원본 dtype 점검 ===')
for col in datetime_cols:
    rd = train_raw[col].dtype
    textlike = pd.api.types.is_object_dtype(rd) or pd.api.types.is_string_dtype(rd)
    ok = textlike or pd.api.types.is_datetime64_any_dtype(rd)
    print(f'{col:45s} 원본 {str(rd):20s}' + ('' if ok else '  <-- 의심(날짜 아님)'))

# 8-2. 최종 train 컬럼 목록
print('\n=== 최종 train 컬럼 목록 ===')
for i, col in enumerate(train.columns):
    print(f'{i:2d}  {col:48s} {train[col].dtype}')

=== datetime_cols 원본 dtype 점검 ===
Customer_registration_datetime                원본 object              
Account_creation_datetime                     원본 object              
Transaction_Datetime                          원본 object              
Last_atm_transaction_datetime                 원본 object              
Last_bank_branch_transaction_datetime         원본 object              
Transaction_resumed_date                      원본 object              

=== 최종 train 컬럼 목록 ===
 0  ID                                               object
 1  Customer_Birthyear                               int64
 2  Customer_Gender                                  object
 3  Customer_personal_identifier                     object
 4  Customer_identification_number                   object
 5  Customer_registration_datetime                   datetime64[ns]
 6  Customer_credit_rating                           object
 7  Customer_flag_change_of_authentication_1         int64
 8  Customer_flag_change_of_authenti

## 8. 금액 부호 분리

In [ ]:
# 전체 수치형 컬럼 중 음수값이 있는 컬럼 조회

numeric_cols = train.select_dtypes(include=["int64", "float64"]).columns

negative_report = []

for col in numeric_cols:
    negative_count = (train[col] < 0).sum()

    if negative_count > 0:
        negative_report.append({
            "column": col,
            "negative_count": negative_count,
            "negative_ratio_percent": round(negative_count / len(train) * 100, 4),
            "min_value": train[col].min()
        })

negative_report = pd.DataFrame(negative_report)

display(negative_report.sort_values("negative_count", ascending=False))

,column,negative_count,negative_ratio_percent,min_value
2,Transaction_Amount,35817,29.8475,-382480000
0,Account_initial_balance,4152,3.4600,-47002364
1,Account_balance,1661,1.3842,-45756563


In [ ]:
# Transaction_Amount 부호 분리 + 절댓값 컬럼 생성

amount_col = "Transaction_Amount"

# 출금/입금 이진 컬럼
# 1 = 출금(음수 거래금액)
# 0 = 입금(0 이상 거래금액)
train["Transaction_is_withdrawal"] = (train[amount_col] < 0).astype(int)

# 거래금액 절댓값 컬럼
train["Transaction_Amount_abs"] = train[amount_col].abs()

In [ ]:
display(
    train[[amount_col, "Transaction_is_withdrawal", "Transaction_Amount_abs"]]
    .head(20)
)

,Transaction_Amount,Transaction_is_withdrawal,Transaction_Amount_abs
0,10000,0,10000
1,-25160000,1,25160000
2,20130000,0,20130000
3,24620000,0,24620000
4,-30000,1,30000
5,29390000,0,29390000
6,17720000,0,17720000
7,-120000,1,120000
8,12980000,0,12980000
9,12640000,0,12640000


In [ ]:
if 'Transaction_Amount' in train.columns:
    train = train.drop(columns=['Transaction_Amount'])
print('남은 컬럼 수:', train.shape[1])

남은 컬럼 수: 65


### 8-5. Location → Location_region 지역 파생

In [ ]:
def extract_region(location):
    if pd.isna(location):
        return "unknown"

    parts = str(location).strip().split()

    if len(parts) >= 1:
        return parts[0]

    return "unknown"


train["Location_region"] = train["Location"].apply(extract_region)

train = train.drop(columns=["Location"])

print(train[["Location_region"]].head())
print("Location 컬럼 존재 여부:", "Location" in train.columns)
print("Location_region 고유값 수:", train["Location_region"].nunique())
print(train["Location_region"].value_counts().head(20))

  Location_region
0             강원도
1            경상북도
2            경상북도
3            경상북도
4            경상남도
Location 컬럼 존재 여부: False
Location_region 고유값 수: 17
Location_region
경기도        18411
경상북도       14692
충청북도       14423
충청남도       12200
전라남도       12181
강원도        11351
경상남도       10077
전라북도        8016
서울특별시       4275
대구광역시       2745
인천광역시       2371
부산광역시       2190
광주광역시       1877
울산광역시       1670
대전광역시       1587
세종특별자치시     1315
제주특별자치도      619
Name: count, dtype: int64


# 9. 학습/검증 분할 확정 (StratifiedGroupKFold, 80/20)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
N_SPLITS = 5

# 타깃 결측 확인
print("Fraud_Type 결측 수:", train["Fraud_Type"].isna().sum())
assert train["Fraud_Type"].notna().all(), "Fraud_Type에 결측이 있습니다."

# sklearn에 넣기 전에 pandas <NA>가 아닌 일반 문자열 배열로 변환
y = train["Fraud_Type"].astype(str).to_numpy()

groups = (
    train["Customer_identification_number"]
    .where(train["Customer_identification_number"].notna(), "__MISSING_CUSTOMER__")
    .astype(str)
    .to_numpy()
)

sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

tr_idx, va_idx = next(sgkf.split(train, y, groups))

df_tr = train.iloc[tr_idx].copy()
df_va = train.iloc[va_idx].copy()

print(f"train: {len(df_tr):,}건 / valid: {len(df_va):,}건 "
      f"(valid 비율 {len(df_va)/len(train)*100:.1f}%)")

Fraud_Type 결측 수: 0
train: 96,140건 / valid: 23,860건 (valid 비율 19.9%)


In [ ]:
# 검증: (1) 고객 누수 0건  (2) 계좌 누수 0건  (3) 13개 클래스 양쪽 분포

customer_overlap = (
    set(df_tr["Customer_identification_number"].dropna())
    & set(df_va["Customer_identification_number"].dropna())
)

account_overlap = (
    set(df_tr["Account_account_number"].dropna())
    & set(df_va["Account_account_number"].dropna())
)

print("고객 누수(교집합):", len(customer_overlap), "명   ->  0 이어야 정상")
print("계좌 누수(교집합):", len(account_overlap), "개   ->  0 이어야 정상")

assert len(customer_overlap) == 0, "고객 누수 발생: train/valid에 같은 고객이 존재합니다."
assert len(account_overlap) == 0, "계좌 누수 발생: train/valid에 같은 계좌가 존재합니다."

dist = pd.DataFrame({
    "train": df_tr["Fraud_Type"].value_counts().sort_index(),
    "valid": df_va["Fraud_Type"].value_counts().sort_index(),
}).fillna(0).astype(int)

dist["valid_ratio_%"] = (
    dist["valid"] / (dist["train"] + dist["valid"]) * 100
).round(1)

display(dist)

missing_valid_classes = dist.index[dist["valid"] == 0].tolist()
print("valid에 0건인 클래스:", missing_valid_classes or "없음")

assert len(missing_valid_classes) == 0, "valid에 존재하지 않는 Fraud_Type 클래스가 있습니다."

고객 누수(교집합): 0 명   ->  0 이어야 정상
계좌 누수(교집합): 0 개   ->  0 이어야 정상


,train,valid,valid_ratio_%
Fraud_Type,,,
a,77,23,23.0
b,78,22,22.0
c,73,27,27.0
d,84,16,16.0
e,77,23,23.0
f,84,16,16.0
g,72,28,28.0
h,80,20,20.0
i,77,23,23.0


valid에 0건인 클래스: 없음
